In [2]:
import pandas as pd
import glob


In [10]:
import glob
import pandas as pd

PATH = r"C:/Users/alexa/OneDrive/Desktop/IDX Internship/raw/"

files = glob.glob(PATH + "CRMLSSold*")

print(f"Number of monthly SOLD files found: {len(files)}")

row_counts = []
for f in files:
    temp = pd.read_csv(f, encoding='latin1')
    row_counts.append((f, len(temp)))

print("\nRow counts BEFORE concatenation:")
for fname, count in row_counts:
    print(f"{fname}: {count:,} rows")

df_sold = pd.concat(
    [pd.read_csv(f, encoding='latin1') for f in files],
    ignore_index=True
)

print(f"\nTotal rows AFTER concatenation: {len(df_sold):,}")

df_res_sold = df_sold[df_sold["PropertyType"] == "Residential"]

print(f"Total rows AFTER Residential filter: {len(df_res_sold):,}")

df_res_sold.to_csv("combined_sold.csv", index=False)
print("\nSaved filtered SOLD dataset to combined_sold.csv")



Number of monthly SOLD files found: 27


C:\Users\alexa\AppData\Local\Temp\ipykernel_12724\314879099.py:12: DtypeWarning: Columns (2,36,39,56,74) have mixed types. Specify dtype option on import or set low_memory=False.
  temp = pd.read_csv(f, encoding='latin1')
C:\Users\alexa\AppData\Local\Temp\ipykernel_12724\314879099.py:12: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  temp = pd.read_csv(f, encoding='latin1')
C:\Users\alexa\AppData\Local\Temp\ipykernel_12724\314879099.py:12: DtypeWarning: Columns (4,74) have mixed types. Specify dtype option on import or set low_memory=False.
  temp = pd.read_csv(f, encoding='latin1')



Row counts BEFORE concatenation:
C:/Users/alexa/OneDrive/Desktop/IDX Internship/raw\CRMLSSold202401.csv: 17,976 rows
C:/Users/alexa/OneDrive/Desktop/IDX Internship/raw\CRMLSSold202402.csv: 19,925 rows
C:/Users/alexa/OneDrive/Desktop/IDX Internship/raw\CRMLSSold202403.csv: 23,276 rows
C:/Users/alexa/OneDrive/Desktop/IDX Internship/raw\CRMLSSold202404.csv: 24,640 rows
C:/Users/alexa/OneDrive/Desktop/IDX Internship/raw\CRMLSSold202405.csv: 26,487 rows
C:/Users/alexa/OneDrive/Desktop/IDX Internship/raw\CRMLSSold202406.csv: 24,328 rows
C:/Users/alexa/OneDrive/Desktop/IDX Internship/raw\CRMLSSold202407.csv: 26,240 rows
C:/Users/alexa/OneDrive/Desktop/IDX Internship/raw\CRMLSSold202408.csv: 24,558 rows
C:/Users/alexa/OneDrive/Desktop/IDX Internship/raw\CRMLSSold202409.csv: 21,267 rows
C:/Users/alexa/OneDrive/Desktop/IDX Internship/raw\CRMLSSold202410.csv: 23,274 rows
C:/Users/alexa/OneDrive/Desktop/IDX Internship/raw\CRMLSSold202411.csv: 20,279 rows
C:/Users/alexa/OneDrive/Desktop/IDX Intern

C:\Users\alexa\AppData\Local\Temp\ipykernel_12724\314879099.py:20: DtypeWarning: Columns (2,36,39,56,74) have mixed types. Specify dtype option on import or set low_memory=False.
  [pd.read_csv(f, encoding='latin1') for f in files],
C:\Users\alexa\AppData\Local\Temp\ipykernel_12724\314879099.py:20: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  [pd.read_csv(f, encoding='latin1') for f in files],
C:\Users\alexa\AppData\Local\Temp\ipykernel_12724\314879099.py:20: DtypeWarning: Columns (4,74) have mixed types. Specify dtype option on import or set low_memory=False.
  [pd.read_csv(f, encoding='latin1') for f in files],



Total rows AFTER concatenation: 587,279
Total rows AFTER Residential filter: 394,150

Saved filtered SOLD dataset to combined_sold.csv


In [11]:
df_res_sold.shape
df_res_sold.info()


<class 'pandas.core.frame.DataFrame'>
Index: 394150 entries, 0 to 587269
Data columns (total 84 columns):
 #   Column                        Non-Null Count   Dtype  
---  ------                        --------------   -----  
 0   BuyerAgentAOR                 345454 non-null  object 
 1   ListAgentAOR                  347965 non-null  object 
 2   Flooring                      252658 non-null  object 
 3   ViewYN                        360505 non-null  object 
 4   WaterfrontYN                  255 non-null     object 
 5   BasementYN                    7723 non-null    object 
 6   PoolPrivateYN                 359983 non-null  object 
 7   OriginalListPrice             393435 non-null  float64
 8   ListingKey                    394150 non-null  int64  
 9   ListAgentEmail                368591 non-null  object 
 10  CloseDate                     394150 non-null  object 
 11  ClosePrice                    394148 non-null  float64
 12  ListAgentFirstName            391191 non-null  ob

In [12]:
print("Initial row count:", len(df_res_sold))
print("Initial column count:", df_res_sold.shape[1])

# 1. DOCUMENT UNIQUE PROPERTY TYPES
print("\nUnique Property Types:")
print(df_sold["PropertyType"].unique())

# 2. NULL COUNT SUMMARY TABLE
null_summary = df_res_sold.isnull().sum().to_frame(name="NullCount")
null_summary["NullPercent"] = (null_summary["NullCount"] / len(df_res_sold)) * 100

print("\nNull Count Summary Table:")
print(null_summary)

# 3. FLAG COLUMNS ABOVE 90% NULL
high_null_cols = null_summary[null_summary["NullPercent"] > 90].index.tolist()

print("\nColumns ABOVE 90% null:")
for col in high_null_cols:
    print(f"- {col}")

# 4. REMOVE COLUMNS ABOVE 90% NULL
df_filtered_sold = df_res_sold.drop(columns=high_null_cols)
print(f"\nColumn count AFTER removing >90% null columns: {df_filtered_sold.shape[1]}")

# 5. NUMERIC DISTRIBUTION SUMMARY
#    For ClosePrice, LivingArea, DaysOnMarket
numeric_cols = ["ClosePrice", "LivingArea", "DaysOnMarket"]

print("\nNumeric Distribution Summary:")
for col in numeric_cols:
    if col in df_filtered_sold.columns:
        print(f"\n--- {col} ---")
        print(df_filtered_sold[col].describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]))
    else:
        print(f"\n--- {col} NOT FOUND IN DATASET ---")

# 6. SAVE FILTERED DATASET
df_filtered_sold.to_csv("filtered_week2_3_sold_output.csv", index=False)
print("\nSaved cleaned dataset to filtered_week2_3_sold_output.csv")

Initial row count: 394150
Initial column count: 84

Unique Property Types:
['Residential' 'CommercialLease' 'Land' 'ResidentialLease'
 'ManufacturedInPark' 'ResidentialIncome' 'CommercialSale'
 'BusinessOpportunity']

Null Count Summary Table:
                             NullCount  NullPercent
BuyerAgentAOR                    48696    12.354687
ListAgentAOR                     46185    11.717620
Flooring                        141492    35.898008
ViewYN                           33645     8.536090
WaterfrontYN                    393895    99.935304
...                                ...          ...
OriginatingSystemSubName        358351    90.917417
BuyerAgencyCompensationType     348014    88.294812
BuyerAgencyCompensation         348025    88.297602
latfilled                       330266    83.791957
lonfilled                       330266    83.791957

[84 rows x 2 columns]

Columns ABOVE 90% null:
- WaterfrontYN
- BasementYN
- FireplacesTotal
- AboveGradeFinishedArea
- TaxAnnualAm

In [13]:

url = "https://fred.stlouisfed.org/graph/fredgraph.csv?id=MORTGAGE30US"

mortgage = pd.read_csv(url, parse_dates=['observation_date'])
mortgage.columns = ['date', 'rate_30yr_fixed']

mortgage['year_month'] = mortgage['date'].dt.to_period('M')

mortgage_monthly = (
    mortgage.groupby('year_month')['rate_30yr_fixed']
    .mean()
    .reset_index()
)

df_filtered_sold['year_month'] = pd.to_datetime(
    df_filtered_sold['ListingContractDate']
).dt.to_period('M')

sold_with_rates = df_filtered_sold.merge(
    mortgage_monthly, on='year_month', how='left'
)

null_rates = sold_with_rates['rate_30yr_fixed'].isnull().sum()
print("Sold rows with NULL mortgage rate:", null_rates)

sold_with_rates.to_csv("sold_with_mortgage_rates.csv", index=False)
print("Saved sold_with_mortgage_rates.csv")


Sold rows with NULL mortgage rate: 1
Saved sold_with_mortgage_rates.csv


In [14]:
pd.read_csv('sold_with_mortage_rates.csv')

FileNotFoundError: [Errno 2] No such file or directory: 'sold_with_mortage_rates.csv'